In [11]:
import os
from dotenv import load_dotenv
from datapizzai.clients import ClientFactory
from datapizzai.tools import tool

# Carica variabili d'ambiente
load_dotenv()

@tool
def calcolatrice(espressione: str) -> str:
    """Esegue calcoli matematici sicuri.
    
    Args:
        espressione: Espressione matematica (es: "2 + 3 * 4")
    
    Returns:
        Risultato del calcolo o messaggio di errore
    """
    try:
        # Validazione sicurezza
        allowed_chars = set('0123456789+-*/(). ')
        if not all(c in allowed_chars for c in espressione):
            return "Errore: Caratteri non permessi"
        
        result = eval(espressione)
        return f"Risultato: {result}"
    except Exception as e:
        return f"Errore: {str(e)}"

In [6]:
def create_calculator_client():
    """Crea un client specializzato in calcoli matematici."""
    
    client = ClientFactory.create(
        provider="openai",                    # Provider AI
        api_key=os.getenv("OPENAI_API_KEY"),  # API key da .env
        model="gpt-4o",                       # Modello OpenAI
        system_prompt="""Sei un assistente matematico esperto.
        Usa sempre lo strumento 'calcolatrice' per eseguire operazioni matematiche.
        Fornisci spiegazioni chiare e dettagliate.""",
        temperature=1,
    )
    
    if not client:
        raise ValueError("❌ Impossibile creare client OpenAI")
    
    return client

In [12]:
# 1. Crea il client
client = create_calculator_client()

# 2. Definisci i tool disponibili
tools = [calcolatrice]

# 3. Esegui query con tool automatico
response = client.invoke(
    input="Sia $k = \lceil{\sqrt{m + n}}\rceil$, dove $n$ e $m$ sono due numeri distinti naturali minori di $100$. Trova il massimo valore di $k$",
    tools=tools,
    tool_choice="auto"  # OpenAI sceglie automaticamente quando usare i tool
)

# 4. Gestisci i risultati
def execute_tool_calls(response, available_tools):
    """Esegue i function call usando i tool passati (non il contenuto testuale)."""
    tool_results = []
    tool_map = {t.name: t for t in available_tools}

    for call in getattr(response, "function_calls", []) or []:
        tool_name = getattr(call, "name", None)
        arguments = getattr(call, "arguments", {}) or {}

        print(f"🔧 Tool chiamato: {tool_name}")
        print(f"📋 Argomenti: {arguments}")

        if tool_name in tool_map:
            result = tool_map[tool_name](**arguments)
            tool_results.append(result)
            print(f"✅ Risultato: {result}")
        else:
            print(f"⚠️ Tool sconosciuto: {tool_name}")
    
    return tool_results

# 5. Esegui i tool e mostra risultati
tool_results = execute_tool_calls(response, tools)

# 6. Mostra risposta finale
if response.text.strip():
    print(f"🤖 Assistente: {response.text}")
elif tool_results:
    print(f"🤖 Assistente: {tool_results[0]}")

<>:9: SyntaxWarning: invalid escape sequence '\l'
<>:9: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_8737/895562917.py:9: SyntaxWarning: invalid escape sequence '\l'
  input="Sia $k = \lceil{\sqrt{m + n}}\rceil$, dove $n$ e $m$ sono due numeri distinti naturali minori di $100$. Trova il massimo valore di $k$",


🤖 Assistente: Per trovare il massimo valore di \( k \), dobbiamo rendere massimo il valore di \( \sqrt{m + n} \).

Poiché \( n \) e \( m \) sono numeri naturali distinti minori di 100, la somma \( m + n \) può essere al massimo 99 + 98. 

Calcoliamo quindi \( k = \lceil \sqrt{99 + 98} \rceil \). Prima determinare il valore della radice quadrata:

\[
\sqrt{99 + 98} = \sqrt{197}
\]

Dopo di ciò, dobbiamo calcolare il valore approssimativo per verificare il massimo valore di \( k \).

Vediamo il calcolo preciso della radice quadrata di 197.


In [13]:
# 1. Definisci strumenti aggiuntivi
@tool
def cerca_informazioni(query: str, max_results: int = 3, lang: str = "it") -> str:
    """Esegue una ricerca web reale (Bing Web Search API o SerpAPI).
    
    Args:
        query: Termine di ricerca
        max_results: Numero massimo di risultati da restituire
        lang: Lingua preferita (es. "it", "en")
    
    Returns:
        Un elenco sintetico di risultati con titolo e URL
    """
    import os
    import requests

    # Scegli motore: SERPAPI o Bing (in base a chiave disponibile)
    serpapi_key = os.getenv("SERPAPI_API_KEY")

    try:
        results = []
        if serpapi_key:
            # SerpAPI Web Search
            params = {
                "engine": "google",
                "q": query,
                "hl": lang,
                "num": max_results,
                "api_key": serpapi_key,
            }
            r = requests.get("https://serpapi.com/search", params=params, timeout=20)
            r.raise_for_status()
            data = r.json()
            for item in (data.get("organic_results") or [])[:max_results]:
                title = item.get("title")
                link = item.get("link")
                if title and link:
                    results.append(f"- {title} — {link}")
        else:
            return (
                "⚠️ Nessuna chiave trovata per la ricerca web. "
                "Configura SERPAPI_API_KEY o BING_SEARCH_API_KEY nel file .env"
            )

        if not results:
            return f"Nessun risultato per '{query}'"
        return "Risultati:\n" + "\n".join(results)

    except Exception as e:
        return f"Errore nella ricerca: {e}"

@tool  
def gestisci_file(comando: str, percorso: str) -> str:
    """Gestisce file e directory in un sistema simulato.
    
    Args:
        comando: Operazione da eseguire (list, create, delete)
        percorso: Percorso del file o directory
    
    Returns:
        Risultato dell'operazione
    """
    # Simulazione file system
    files_system = {
        "docs/": ["README.md", "guide.txt"],
        "src/": ["main.py", "utils.py"],
        "data/": ["dataset.csv", "config.json"]
    }
    
    if comando == "list":
        if percorso in files_system:
            files = files_system[percorso]
            return f"Contenuto di {percorso}:\n" + "\n".join(f"- {f}" for f in files)
        return f"Directory {percorso} non trovata"
    
    elif comando == "create":
        return f"File {percorso} creato con successo"
    
    elif comando == "delete":
        return f"File {percorso} eliminato con successo"
    
    return f"Comando '{comando}' non supportato"

# 2. Crea client multi-tool
def create_multi_tool_client():
    """Crea un client con accesso a tutti gli strumenti."""
    
    client = ClientFactory.create(
        provider="openai",
        api_key=os.getenv("OPENAI_API_KEY"),
        model="gpt-4o",
        system_prompt="""Sei un assistente AI versatile con accesso a strumenti specializzati:

        - calcolatrice: per operazioni matematiche
        - cerca_informazioni: per ricerche web simulate  
        - gestisci_file: per operazioni su file e directory

        Analizza ogni richiesta e scegli lo strumento più appropriato.
        Per task complessi, puoi usare più strumenti in sequenza.
        Spiega sempre cosa stai facendo e perché."""
    )
    
    return client

# 3. Configura tutti i tool
tools = [calcolatrice, cerca_informazioni, gestisci_file]

# 4. Esegui workflow complessi
client = create_multi_tool_client()

complex_query = """
Esegui questo workflow:
1. Cerca informazioni su machine learning
2. Calcola quanti anni sono passati dal 1990 al 2025
3. Crea un file chiamato ml_summary.txt nella directory docs/
4. Lista i file nella directory docs/ per verificare
"""

response = client.invoke(
    input=complex_query,
    tools=tools,
    tool_choice="auto"
)

# Il modello OpenAI sceglierà automaticamente i tool necessari
tool_results = execute_tool_calls(response, tools)

🔧 Tool chiamato: cerca_informazioni
📋 Argomenti: {'query': 'machine learning', 'max_results': 3, 'lang': 'it'}
✅ Risultato: Risultati:
- Machine learning — https://en.wikipedia.org/wiki/Machine_learning
- What Is Machine Learning (ML)? — https://www.ibm.com/think/topics/machine-learning
- Machine Learning Crash Course — https://developers.google.com/machine-learning/crash-course
🔧 Tool chiamato: calcolatrice
📋 Argomenti: {'espressione': '2025 - 1990'}
✅ Risultato: Risultato: 35
